# 实验 6：Strands Agents 与 Bedrock AgentCore Memory

本实验演示如何将 Strands Agents 与 Amazon Bedrock AgentCore Memory 集成，以创建具有跨对话和会话持久记忆能力的 AI 代理。

## 学习目标

完成本实验后，您将能够：
- 了解 Bedrock AgentCore Memory 的功能和架构
- 实现短期和长期记忆策略
- 跨会话存储和检索对话上下文
- 自动提取用户偏好和语义事实
- 构建具有记忆功能的代理以提供个性化体验
- 在真实对话中测试记忆检索和上下文注入

## 前提条件

在开始本实验之前，请确保您已具备：
- 已配置 AWS 凭证（IAM 角色或环境变量）
- 已安装所需的 Python 包
- 基于 AWS 区域的 Nova Pro 模型 ID

如果您未在已承担 IAM 角色的环境中运行，请将 AWS 凭证设置为环境变量：

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

安装 Strands Agents 和 Bedrock AgentCore Python SDK 所需的包：

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

根据 AWS 区域设置 Nova Pro 模型 ID：

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## 演示 Strands Agents 的记忆限制

默认情况下，Strands Agents **没有内置的长期持久记忆**。每次对话都是从零开始，无法访问之前的交互或已学习的上下文。让我们先演示这一限制，然后展示 AgentCore Memory 如何解决这一挑战。

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

# 创建一个用于演示的自定义天气工具
@tool
def weather(city: str) -> str:
    """获取城市的天气信息
    Args:
        city: 城市或地点名称
    """
    return f"{city}的天气：晴天，35°C"  # 用于演示的虚拟结果

# Create your first agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="你是一个生活助手，运用科学的知识回答各种问题。",
    tools=[weather],
)

agent("我们之前聊了什么？")

## 什么是 Bedrock AgentCore Memory？

### 短期记忆：

短期记忆捕获原始交互事件，维护即时上下文，支持实时对话，丰富长期记忆系统，并能够构建高级上下文解决方案，例如多步骤任务完成、会话内知识积累和上下文感知决策。

- **同步存储** - 消息在对话过程中立即保存

- **可配置的保留期** - 在指定时间段后自动过期（7 到 365 天）


### 长期记忆：

长期记忆存储从原始代理交互中提取的结构化信息，这些信息在多个会话之间保留。长期记忆不会保存所有原始对话数据，而是仅保留关键洞察，例如对话摘要、事实和知识（语义记忆）或用户偏好。

记忆管道是一个在后台运行的异步过程，在原始对话/上下文通过 CreateEvent 存储到短期记忆后，它会自动提取洞察。这种方式可以高效地整合关键信息，而不会中断实时交互。

![memory-pipeline.png](images/memory-pipeline.png)

## 创建 AgentCore Memory

让我们创建一个同时具有短期和长期记忆功能的 AgentCore Memory 实例。此过程大约需要 **3 分钟**来设置向量数据库和处理管道。

我们将配置三种记忆策略：
- **摘要策略** - 生成会话摘要以实现高效的上下文压缩
- **用户偏好策略** - 捕获并存储用户行为模式
- **语义策略** - 从对话中提取并存储事实知识

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from botocore.exceptions import ClientError
import boto3

region = boto3.session.Session().region_name

memory_client = MemoryClient(region_name=region)
memory_name = "SampleAgentMemory"

try:
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        description="Memory for sample agent conversations",
        strategies=[
            {
                StrategyType.SUMMARY.value: {
                    "name": "SessionSummarizer",
                    "namespaces": ["sample-agent/summaries/{actorId}/{sessionId}"]
                }
            },
            {
                StrategyType.USER_PREFERENCE.value: {
                    "name": "UserPreferences",
                    "description": "Captures user preferences and behavior",
                    "namespaces": ["sample-agent/preferences/{actorId}"],
                }
            },
            {
                StrategyType.SEMANTIC.value: {
                    "name": "FactExtractor",
                    "description": "Stores facts from conversations",
                    "namespaces": ["sample-agent/semantic/{actorId}/"],
                }
            },
        ],
        event_expiry_days=7, # Memories expire after 7 days
    )
    memory_id = memory.get('id') # The memory_id will be used in following operations
    print(f"✅ Memory ID: {memory_id}")
except ClientError as e:
    print(f"❌ ERROR: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        print(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Show any errors during memory creation
    print(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            print(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            print(f"Failed to clean up memory: {cleanup_error}")

## 存储对话事件

现在让我们演示如何在 AgentCore Memory 中存储对话事件。我们将创建多个对话交互，以便用不同类型的内容填充短期和长期记忆。

### 示例对话：旅行规划助手

我们将模拟一个全面的旅行规划对话，其中用户询问有关香港活动的信息。此对话将演示 AgentCore Memory 如何捕获：

- **会话上下文**（对话流程和讨论的主题）
- **用户偏好**（徒步旅行、当地美食体验、适合家庭的活动）
- **语义事实**（关于 Bedrock AgentCore 的信息）


以下对话事件将用丰富多样的内容填充短期和长期记忆，以测试记忆检索功能。

In [ ]:
# Simulate a conversation session with Hong Kong activities enquiry

USER_ID = "123"
SESSION_ID = "12345-67890"

memory_client.create_event(
    memory_id=memory_id, # This is the id from create_memory or list_memories
    actor_id=USER_ID,  # This is the identifier of the actor, could be an agent or end-user.
    session_id=SESSION_ID, #Unique id for a particular request/conversation.
    messages=[
        ("你推荐在香港有哪些活动？", "USER"),
        ("""香港提供了令人难以置信的丰富体验！以下是一些值得考虑的热门活动：

        ## 必看景点
        - **太平山顶** - 乘坐历史悠久的山顶缆车欣赏壮丽的城市景观
        - **天星小轮** - 往返尖沙咀和中环的标志性海港渡轮
        - **幻彩咏香江** - 维多利亚港上空的每晚多媒体灯光秀
        - **天坛大佛和宝莲禅寺** - 乘坐缆车前往大屿山的巨型青铜大佛

        ## 美食与文化
        - **点心** - 尝试传统茶楼或现代餐厅
        - **街头美食市场** - 庙街夜市、女人街
        - **天台酒吧** - 一边品鸡尾酒一边欣赏天际线美景
        - **传统寺庙** - 文武庙、黄大仙祠

        ## 独特体验
        - **电车之旅** - 乘坐历史悠久的"叮叮车"穿越城市
        - **龙脊远足径** - 拥有海岸美景的风景步道
        - **万佛寺** - 氛围独特的寺庙建筑群
        - **星光大道** - 庆祝香港电影的海滨长廊

        ## 一日游
        - **澳门** - 乘坐渡轮轻松前往，体验葡萄牙文化和赌场
        - **离岛** - 南丫岛、长洲，享受海滩和海鲜
        """, "ASSISTANT")
    ]
)

In [ ]:
# User preference with hiking activity and ask for Hong Kong hiking trails

USER_ID = "123"
SESSION_ID = "12345-67890"

memory_client.create_event(
    memory_id=memory_id, # This is the id from create_memory or list_memories
    actor_id=USER_ID,  # This is the identifier of the actor, could be an agent or end-user.
    session_id=SESSION_ID, #Unique id for a particular request/conversation.
    messages=[
        ("我喜欢远足！你能推荐更多远足路线吗？", "USER"),
        ("""太好了！香港拥有世界上最好的城市远足路线，有着令人惊叹的海岸和山地步道。以下是我的推荐：

        ## 标志性路线
        - **龙脊** - 港岛最著名的山脊远足路线，拥有全景海洋美景
        - **狮子山** - 具有挑战性的攀登，拥有令人难以置信的 360° 城市景观（香港精神的象征！）
        - **太平山环回步行径** - 环绕太平山顶的轻松铺装步道，可欣赏海港景色
        - **紫罗兰山** - 俯瞰浅水湾的绝佳日落观赏点

        ## 海岸美景
        - **麦理浩径第二段** - 西贡令人惊叹的海岸线和海滩
        - **大浪湾** - 通过风景优美的远足路线可到达的偏远海滩
        - **鹤咀** - 港岛东南端崎岖的海岸线
        - **桥咀洲** - 独特的连岛沙洲连接两座岛屿

        ## 山地探险
        - **大东山** - 大屿山第二高峰，以日出/日落闻名
        - **大帽山** - 香港最高峰（957米），拥有云雾缭绕的山景
        - **马鞍山** - 独特的金字塔形山峰，攀登具有挑战性
        - **钓鱼翁** - 西贡最高点，拥有 270° 海景
        """, "ASSISTANT"),
    ],
)

In [ ]:
# Provide the background of family with kids and ask for child friendly spots

USER_ID = "123"
SESSION_ID = "12345-67890"

memory_client.create_event(
    memory_id=memory_id, # This is the id from create_memory or list_memories
    actor_id=USER_ID,  # This is the identifier of the actor, could be an agent or end-user.
    session_id=SESSION_ID, #Unique id for a particular request/conversation.
    messages=[
        ("我家里有小孩。", "USER"),
        ("""香港有几条适合亲子的远足路线，兼具探险乐趣和适中的难度：

        **最适合家庭的轻松路线：**
        - **港岛径第一段（太平山环回步行径）** - 平坦铺装的 3.5 公里环形步道，景色优美，附近有游乐场（1 小时）
        - **长洲家庭步道** - 环绕这座无车岛屿的平缓小径，有海滩和小吃店（2 小时，可缩短）
        - **南丫岛家庭步行径** - 连接村庄之间的轻松步道，有海滩、游乐场和海鲜餐厅（2-3 小时）

        **短途探险：**
        - **新娘潭自然教育径** - 1 公里步行至美丽瀑布，设有野餐区（30 分钟加游玩时间）
        - **龙虎山晨运径** - 通往太平山顶的短途上坡路，设有访客中心和花园（1 小时）
        - **大潭树木研习径** - 寓教于乐的树木观赏步道，可欣赏水塘景色（1 小时）

        **家庭远足小贴士：**
        - 尽早出发以避开正午高温
        - 携带充足的零食、饮用水和防晒用品
        - 选择有洗手间设施的路线
        - 规划靠近公共交通的路线
        - 考虑携带驱虫剂

        您想根据孩子的年龄获取具体推荐吗？
        """, "ASSISTANT"),
    ],
)

In [ ]:
# Interested in dim sun and follow up enquiry for dim sum restaurant recommendation

USER_ID = "123"
SESSION_ID = "12345-67890"

memory_client.create_event(
    memory_id=memory_id, # This is the id from create_memory or list_memories
    actor_id=USER_ID,  # This is the identifier of the actor, could be an agent or end-user.
    session_id=SESSION_ID, #Unique id for a particular request/conversation.
    messages=[
        ("我想尝试点心。你能推荐一些香港著名的点心餐厅吗？", "USER"),
        ("""当然！香港是世界点心之都。以下是一些必去的餐厅：

        ## 米其林星级餐厅
        - **添好运** - 世界上最便宜的米其林星级餐厅，以叉烧包闻名
        - **龙景轩** - 四季酒店内，高档粤菜，可欣赏海港景色
        - **都爹利会馆** - 坐落于历史建筑中的优雅环境，现代粤菜

        ## 传统茶楼
        - **美心皇宫** - 大会堂内的经典推车服务，老派氛围
        - **莲香楼** - 始于 1920 年代的正宗茶楼，从流动推车上取菜
        - **陆羽茶室** - 创立于 1933 年的历史老店，传统装潢

        ## 现代人气餐厅
        - **南华茶室** - 时尚餐厅，摆盘精美适合拍照
        - **一点心** - 多家分店，性价比高且品质优良
        - **点心广场** - 以创意摆盘呈现经典点心的现代风格
        """, "ASSISTANT"),
    ],
)

In [ ]:
# Simulate another session with customer support about product return

USER_ID = "123"
SESSION_ID = "67890-12345"

memory_client.create_event(
    memory_id=memory_id, # This is the id from create_memory or list_memories
    actor_id=USER_ID,  # This is the identifier of the actor, could be an agent or end-user.
    session_id=SESSION_ID, #Unique id for a particular request/conversation.
    messages=[
        ("我正在计划构建一个旅行智能体。什么是 Amazon Bedrock AgentCore？", "USER"),
        ("""Amazon Bedrock AgentCore 使您能够使用任何框架和模型，安全地大规模部署和运行高效的智能体。借助 Amazon Bedrock AgentCore，开发者可以以实际部署所需的规模、可靠性和安全性加速将 AI 智能体投入生产。AgentCore 提供工具和功能使智能体更高效、更强大，提供专门构建的基础设施以安全地扩展智能体，并提供控制措施以运行可信赖的智能体。Amazon Bedrock AgentCore 服务是可组合的，可与流行的开源框架和任何模型配合使用，因此您无需在开源灵活性和企业级安全性及可靠性之间做出选择。

        Amazon Bedrock AgentCore 包含以下模块化服务，您可以一起使用或独立使用：

        * Amazon Bedrock AgentCore Runtime
        AgentCore Runtime 是一个安全的无服务器运行时，专为使用任何开源框架（包括 LangGraph、CrewAI 和 Strands Agents）、任何协议和任何模型来部署和扩展动态 AI 智能体和工具而构建。Runtime 专为智能体工作负载而设计，具有业界领先的扩展运行时支持、快速冷启动、真正的会话隔离、内置身份验证以及对多模态负载的支持。开发者可以专注于创新，而 Amazon Bedrock AgentCore Runtime 负责处理基础设施和安全性，从而加速产品上市时间。

        * Amazon Bedrock AgentCore Identity
        AgentCore Identity 提供安全、可扩展的智能体身份和访问管理功能，加速 AI 智能体开发。它与现有身份提供商兼容，无需用户迁移或重建身份验证流程。AgentCore Identity 通过安全的令牌保管库帮助减少授权疲劳，并允许您构建流畅的 AI 智能体体验。恰到好处的访问权限和安全的权限委托使智能体能够安全地访问 AWS 资源以及第三方工具和服务。

        * Amazon Bedrock AgentCore Memory
        AgentCore Memory 使开发者能够轻松构建上下文感知的智能体，消除复杂的记忆基础设施管理，同时提供对 AI 智能体记忆内容的完全控制。Memory 提供业界领先的准确性，同时支持用于多轮对话的短期记忆和可在智能体和会话之间共享的长期记忆。

        * Amazon Bedrock AgentCore Code Interpreter
        AgentCore Code Interpreter 工具使智能体能够在隔离的沙箱环境中安全地执行代码。它提供高级配置支持以及与流行框架的无缝集成。开发者可以为复杂的工作流和数据分析构建强大的智能体，同时满足企业安全要求。

        * Amazon Bedrock AgentCore Browser
        AgentCore Browser 工具提供快速、安全的基于云的浏览器运行时，使 AI 智能体能够大规模地与网站交互。它提供企业级安全性、全面的可观测性功能，并自动扩展——所有这些都无需基础设施管理开销。

        * Amazon Bedrock AgentCore Gateway
        Amazon Bedrock AgentCore Gateway 为智能体提供了一种安全的方式来发现和使用工具，并能轻松地将 API、Lambda 函数和现有服务转换为智能体兼容的工具。Gateway 消除了数周的自定义代码开发、基础设施配置和安全实施工作，使开发者能够专注于构建创新的智能体应用程序。

        * Amazon Bedrock AgentCore Observability
        AgentCore Observability 帮助开发者通过统一的运营仪表板在生产环境中追踪、调试和监控智能体性能。通过支持 OpenTelemetry 兼容的遥测数据和智能体工作流每个步骤的详细可视化，AgentCore 使开发者能够轻松了解智能体行为并大规模维护质量标准。
        """, "ASSISTANT")
    ],
)

## 记忆检索与分析

存储对话事件后，AgentCore Memory 会自动处理它们以提取洞察。让我们探索如何检索不同类型的记忆：

### 短期记忆检索

短期记忆提供对近期对话历史的即时访问。这对于在会话中维护上下文非常有用。

In [ ]:
from rich.table import Table
import rich

console = rich.get_console()

USER_ID = "123"
SESSION_ID = "12345-67890"

events = memory_client.list_events(
    memory_id=memory.get("id"),
    actor_id=USER_ID,
    session_id=SESSION_ID,
    max_results=2,
)

console.print(f"\n会话短期记忆：{SESSION_ID}（窗口大小 = {len(events)}）")
console.rule()

table = Table(title="短期记忆事件", show_lines=True)
table.add_column("#", style="cyan")
table.add_column("时间戳", style="blue")
table.add_column("角色", style="green")
table.add_column("内容", style="magenta")

for i, event in enumerate(events, 1):
    timestamp = str(event['eventTimestamp'])
    for j, payload in enumerate(event['payload']):
        role = payload['conversational']['role']
        content = payload['conversational']['content']
        # 仅在每个事件的第一行显示事件编号和时间戳
        table.add_row(
            str(i) if j == 0 else "",
            timestamp if j == 0 else "",
            role,
            str(content)
        )

console.print(table)


### 通过查询检索长期记忆

长期记忆会自动从对话中提取并存储洞察。短期记忆处理为长期记忆大约需要 **20-30 秒**。

In [ ]:
# Wait 1 minute for Long-Term Memory to be ready...
import time
time.sleep(60)

列出长期记忆中所有策略的记忆记录：

In [ ]:
from rich.table import Table
import rich

console = rich.get_console()

# Fetch memory records
agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
summaries = agentcore_client.list_memory_records(
    memoryId=memory_id,
    namespace="sample-agent/summaries/123",
    maxResults=100
)['memoryRecordSummaries']
preferences = agentcore_client.list_memory_records(
    memoryId=memory_id,
    namespace="sample-agent/preferences/123",
    maxResults=100
)['memoryRecordSummaries']
facts = agentcore_client.list_memory_records(
    memoryId=memory_id,
    namespace="sample-agent/semantic/123",
    maxResults=100
)['memoryRecordSummaries']

records = summaries + preferences + facts

console.print(f"\nTotal Long Term Memory Records: {len(records)}")
console.rule()

table = Table(title="Long Term Memory", show_lines=True)
table.add_column("Strategy ID", style="green")
table.add_column("Content", style="magenta")
table.add_column("Created At", style="blue")

for record in records:
    table.add_row(
        record['memoryStrategyId'],
        record['content']['text'],
        record['createdAt'].strftime("%Y-%m-%d %H:%M:%S")
    )

console.print(table)

让我们探索不同类型的长期记忆：

#### 摘要策略
检索会话摘要，将对话上下文压缩为关键主题和结果。

In [ ]:
USER_ID = "123"

summaries = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"sample-agent/summaries/{USER_ID}/",
    query="我之前问了关于香港的什么问题？",
    top_k=3
)

i=1
for summary in summaries:
    print(f"第 {i} 条结果（Score：{summary['score']}）")
    print("----------------")
    print(f"创建时间: {summary['createdAt']}")
    print(f"內容: {summary['content']['text']}")
    i=i+1

In [ ]:
USER_ID = "123"

summaries = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"sample-agent/summaries/{USER_ID}/",
    query="我之前问了关于 Amazon Bedrock AgentCore 的什么问题？",
    top_k=3
)

i=1
for summary in summaries:
    print(f"第 {i} 条结果（Score：{summary['score']})")
    print("----------------")
    print(f"创建时间: {summary['createdAt']}")
    print(f"內容: {summary['content']['text']}")
    i=i+1

#### 用户偏好策略
根据对话历史捕获并存储用户行为模式和偏好。

In [ ]:
USER_ID = "123"

preferences = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"sample-agent/preferences/{USER_ID}",
    query="有什么日本旅行的推荐吗？", # Semantic search for relevant user preferences based on query
    top_k=3
)

i=1
for preference in preferences:
    print(f"第 {i} 条结果（Score：{preference['score']})")
    print("----------------")
    print(f"创建时间: {preference['createdAt']}")
    print(f"內容: {preference['content']['text']}")
    i=i+1

#### 语义事实策略
从对话中提取并存储事实知识以供将来参考。

⚠️ 语义记忆仅从用户消息中提取信息。

参考：https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-system-prompt.html

In [ ]:
USER_ID = "123"

semantics = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"sample-agent/semantic/{USER_ID}",
    query="什么是 Amazon Bedrock AgentCore？", # Semantic search for fact and domain specific information based on query
    top_k=3
)

i=1
for semantic in semantics:
    print(f"第 {i} 条结果（Score：{semantic['score']})")
    print("----------------")
    print(f"创建时间: {semantic['createdAt']}")
    print(f"內容: {semantic['content']['text']}")
    i=i+1

⚠️ 正如预期，语义事实结果中不包含关于 Bedrock AgentCore 的记忆信息，因为语义记忆仅从用户消息中提取信息。像这样的事实性或快速变化的主题不应依赖长期记忆，而应使用基于工具的检索（如网络搜索）来获取最新信息。如果您仍然想捕获 AgentCore 的问答内容，可以从会话摘要中检索。

## 构建具有记忆功能的代理

现在让我们创建一个 Strands Agent，它会自动使用 AgentCore Memory 根据对话历史提供个性化响应。

让我们构建一个带有 Memory Hook 的 Strands Agent，以与 AgentCore Memory 集成。

![memory-integration-with-agent.png](images/memory-integration-with-agent.png)

In [ ]:
import uuid

from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

region = boto3.session.Session().region_name

USER_ID = "123"
SESSION_ID = str(uuid.uuid4())

agentcore_memory_config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=SESSION_ID,
    actor_id=USER_ID,
    retrieval_config={
        "sample-agent/preferences/{actorId}": RetrievalConfig(
            top_k=5,
            relevance_score=0.7
        ),
        "sample-agent/semantic/{actorId}": RetrievalConfig(
            top_k=10,
            relevance_score=0.3
        ),
        "sample-agent/summaries/{actorId}/{sessionId}": RetrievalConfig(
            top_k=5,
            relevance_score=0.5
        )
    }
)

# Create session manager
session_manager = AgentCoreMemorySessionManager(
    agentcore_memory_config=agentcore_memory_config,
    region_name=region
)

# 创建一个用于演示的自定义天气工具
@tool
def weather(city: str) -> str:
    """获取城市的天气信息
    Args:
        city: 城市或地点名称
    """
    return f"{city}的天气：下雨天，35°C"  # 用于演示的虚拟结果

# Create the customer support agent with all 5 tools
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="你是一个生活助手，运用科学的知识回答各种问题。",
    session_manager=session_manager,
    tools=[weather],
)

### 测试集成了 AgentCore Memory 的 Strands Agent

我们将在多个场景中测试具有记忆功能的代理的能力。

1. **对话历史回忆**：验证代理能否访问并总结之前的对话会话
  
2. **上下文感知的工具集成**：测试存储的偏好如何与实时工具数据结合
  
3. **语义知识检索**：测试从存储的对话中检索事实知识

In [ ]:
# Test the capability of Agent with memory to the past conversation sessions
agent("我之前问了关于香港的什么问题？")

In [ ]:
# Test the capability of Agent with memory and tool as context to provide more precise suggestion based on user preference and external factor
agent("根据今天的天气，有什么推荐在香港做的事情吗？")

In [ ]:
# Test the capability of Agent with retrieving the semantic fact in long-term memory
agent("Bedrock AgentCore Memory 能处理长期记忆吗？")

让我们查看 Strands Agents 如何与 AgentCore Memory 集成并检索长期记忆作为上下文。

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

## [高级] 使用基于 Hook 的架构自定义 Strands Agents 记忆集成

Strands Agents 提供了基于 Hook 的架构，允许您在关键生命周期节点拦截和增强代理行为。通过实现 HookProvider 接口并为 MessageAddedEvent（预处理）和 AfterInvocationEvent（后处理）等事件注册回调，您可以将 AgentCore Memory 无缝集成到代理工作流中。

### 记忆集成架构

我们的记忆增强代理使用基于 Hook 的架构，与 AgentCore Memory 无缝集成：

#### 预处理 Hook（`retrieve_user_context`）
- **触发时机**：在处理每个用户查询之前
- **操作**：从所有记忆策略中检索相关上下文
- **结果**：将个性化上下文注入代理的提示中

#### 后处理 Hook（`save_conversation`）
- **触发时机**：在生成代理响应之后
- **操作**：将交互保存到 AgentCore Memory
- **结果**：实现持续学习和上下文构建

这种架构确保每次交互都能受益于历史上下文，同时为代理不断增长的知识库做出贡献。

In [ ]:
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
import re
import logging

logger = logging.getLogger(__name__)
logging.getLogger().setLevel(logging.ERROR) # Set the logging level to ERROR

class LongTermMemoryHooks(HookProvider):
    """Memory hooks for long-term memory agent"""

    def __init__(
        self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str
    ):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register user support memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_user_context)
        registry.add_callback(AfterInvocationEvent, self.save_conversation)
        logger.info("User support memory hooks registered")

    def retrieve_user_context(self, event: MessageAddedEvent):
        """Retrieve user context before processing support query"""
        logger.info("Start to retrieve user context...")
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]

            try:
                all_context = []

                for context_type, namespace in self.namespaces.items():
                    # *** AGENTCORE MEMORY USAGE *** - Retrieve customer context from each namespace
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id, sessionId=""),
                        query=user_query,
                        top_k=3,
                    )
                    # Post-processing: Format memories into context strings
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(
                                        f"[{context_type.upper()}] {text}"
                                    )

                # Inject user context into the query
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0][
                        "text"
                    ] = f"User Context: {context_text}\n\n User Query: {original_text}"
                    logger.info(f"Retrieved {len(all_context)} user context items")

            except Exception as e:
                logger.error(f"Failed to retrieve user context: {e}")

    def save_conversation(self, event: AfterInvocationEvent):
        """Save user interaction after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last user query and agent response
                user_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        output_message = msg["content"][0]["text"]
                        agent_response = re.sub(r'<thinking>.*?</thinking>', '', output_message, flags=re.DOTALL).strip()

                    elif (
                        msg["role"] == "user"
                        and not user_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        input_prompt = msg["content"][0]["text"]
                        user_query = re.sub(r'User Context:.*? User Query: ', '', input_prompt, flags=re.DOTALL).strip()
                        break

                if user_query and agent_response:
                    # *** AGENTCORE MEMORY USAGE *** - Save the support interaction
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[
                            (user_query, "USER"),
                            (agent_response, "ASSISTANT"),
                        ],
                    )
                    logger.info(f"Saved support interaction to memory")

        except Exception as e:
            logger.error(f"Failed to save support interaction: {e}")


现在让我们创建一个带有 **memory hook** 的 Strands Agent，它会自动使用 AgentCore Memory 根据对话历史提供个性化响应。

In [ ]:
import uuid

from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.memory import MemoryClient
import boto3

region = boto3.session.Session().region_name

memory_client = MemoryClient(region_name=region)

USER_ID = "123"
SESSION_ID = str(uuid.uuid4())
memory_hooks = LongTermMemoryHooks(memory_id, memory_client, USER_ID, SESSION_ID)

# 创建一个用于演示的自定义天气工具
@tool
def weather(city: str) -> str:
    """获取城市的天气信息
    Args:
        city: 城市或地点名称
    """
    return f"{city}的天气：下雨天，35°C"  # 用于演示的虚拟结果

# Create the customer support agent with all 5 tools
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="你是一个生活助手，运用科学的知识回答各种问题。",
    hooks=[memory_hooks], # Pass Memory Hooks
    tools=[weather],
)

In [ ]:
# Test the capability of Agent with memory and tool as context to provide more precise suggestion based on user preference and external factor
agent("根据今天的天气，有什么推荐在香港做的事情吗？")

## 资源清理（可选）

清理 AgentCore Memory 资源以避免不必要的费用：

In [ ]:
import boto3
import os

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)

try:
    print("Deleting AgentCore Memory...")
    agentcore_control_client.delete_memory(memoryId=memory_id)
    print("✓ AgentCore Memory deletion initiated")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

## 总结

在本实验中，您成功完成了以下内容：

- ✅ 创建并配置了具有向量存储的 Bedrock AgentCore Memory
- ✅ 存储了对话事件并构建了全面的记忆数据集
- ✅ 实现了多种记忆检索策略（摘要、偏好、语义事实）
- ✅ 将 AgentCore Memory 与 Strands Agents 集成以实现持久对话上下文
- ✅ 测试了具有跨会话对话连续性的记忆增强代理

## 主要优势

- **持久上下文**：在多个会话和交互之间维护对话历史
- **智能检索**：多种策略用于提取相关上下文（摘要、偏好、事实）
- **可扩展存储**：基于向量的存储，实现高效的相似性搜索和检索
- **无缝集成**：与 Strands Agents 和其他 AI 框架的原生集成
- **灵活策略**：可配置的记忆策略，适用于不同的用例和应用场景